In [ ]:
"""
KORF (Norfolk International Airport) ASOS data download, 2015-2025

Data source: Iowa Environmental Mesonet (IEM), Iowa State University
https://mesonet.agron.iastate.edu/request/download.phtml
Original observations: NWS/FAA Automated Surface Observing System (ASOS)

AI usage disclosure:
Portions of this script were developed with assistance from Claude
(Opus 5.5), a large language model by Anthropic, accessed via
https://claude.ai on October 5, 2026. Claude was used to:
  - identify IEM as an alternative source after the NWS Time Series
    Viewer's download feature was unavailable
  - write the code that builds the IEM request URL and downloads the
    data one year at a time
  - suggest unit conversions (knots to m/s)
The author reviewed, tested, and verified the code and spot-checked
the output against the NWS Time Series Viewer.

Reference:
Anthropic. (2026). Claude (Opus 5.5) [Large language model].
https://claude.ai

-----------------------------------------------------------------------
AI interaction log (Claude Opus 5.5, Anthropic, claude.ai, October 5, 2026)

1. Prompt: How to extract historical data from the NWS Time Series
   Viewer (site=KORF).
   Answer: The page's download feature was unavailable and its table is
   loaded by JavaScript, so scraping isn't practical. Suggested the Iowa
   Environmental Mesonet (IEM) ASOS archive and gave pandas code.

2. Prompt: Shared NWS URL for January 1-30, 2015 in metric units.
   Answer: Provided an equivalent IEM request (station ORF, Eastern time,
   routine and special reports, selected variables).
   
3. Prompt: What units is sknt in?
   Answer: Knots. Conversions: x0.5144 = m/s, x1.852 = km/h, x1.151 = mph.

4. Prompt: Need data from January 2015 to December 2025.
   Answer: Provided a loop that downloads one year at a time, combines
   the years, adds wind_ms/gust_ms, and saves to CSV.
"""

In [5]:
import time
import pandas as pd

base = (
    "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
    "station=ORF&data=tmpc&data=dwpc&data=relh&data=sknt&data=drct"
    "&data=gust&data=mslp&data=p01m&data=vsby&data=wxcodes"
    "&tz=America/New_York&format=onlycomma&latlon=no&missing=M"
    "&trace=0.0001&report_type=3&report_type=4"
)
#Iowa Environmental Mesonet. (2026). ASOS/AWOS/METAR data download: Station ORF (Norfolk International Airport, VA), 1 January 2015 – 31 December 2025 [Data set]. Iowa State University. Retrieved October 5, 2026, from https://mesonet.agron.iastate.edu/request/download.phtml
frames = []
for year in range(2015, 2026):
    url = f"{base}&year1={year}&month1=1&day1=1&year2={year+1}&month2=1&day2=1"
    print(f"Downloading {year}...")
    frames.append(pd.read_csv(url, na_values=["M"], parse_dates=["valid"]))
    time.sleep(2)  # short pause between requests to go easy on the server

df = pd.concat(frames, ignore_index=True)
df["wind_ms"] = df["sknt"] * 0.5144
df["gust_ms"] = df["gust"] * 0.5144
df.to_csv("KORF_2015-2025.csv", index=False)
print(df.shape, df["valid"].min(), df["valid"].max())



(120492, 14) 2015-01-01 00:51:00 2025-12-31 10:51:00
